# Screening activity matters: Evidence from ESG portfolio performance from an emerging market

**Authors**: Ved Dilip Beloskar, S.V.D. Nageswara Rao

**Published**: 2023-02-16

**URL**: [https://doi.org/10.1002/ijfe.2798](https://doi.org/10.1002/ijfe.2798)

**Abstract**: Socially Responsible Investments (SRI) have recently generated much interest among asset owners, managers and academicians. Though the Efficient Market Theory suggests that stock prices fully reflect all available information, few existing studies indicate that Environmental, Social and Governance (ESG) portfolios deliver superior risk‐adjusted performance. ESG investing is at a nascent stage in India but is growing rapidly, especially after the COVID‐19 pandemic. Asset managers always face the dilemma of choosing between different screening methods, screening intensities and stock weighting schemes to deliver outperformance. Our study attempts to investigate the impact of these portfolio construction criteria on the risk‐adjusted performance of ESG portfolios in India. Our results show that there exists a trade‐off between superior investment performance and unsystematic risk of ESG portfolios. Investors can benefit from investing in equally‐weighted best‐in‐class portfolios constructed using ESG scores. We highlight the implications of our findings for asset owners, managers, index providers and regulators, and also provide directions for future research in the area of ESG portfolio management.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

In [ ]:
# Phase 1: Configuration
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration parameters
TICKER = 'AAPL'
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'

In [ ]:
# Phase 2: Data Download and Feature Engineering
data = yf.download(TICKER, start=START_DATE, end=END_DATE)
data['Returns'] = data['Adj Close'].pct_change()
data.dropna(inplace=True)

# Feature: Rolling mean and standard deviation
data['RollingMean'] = data['Adj Close'].rolling(window=20).mean()
data['RollingStd'] = data['Adj Close'].rolling(window=20).std()

In [ ]:
# Phase 3: Signal Generation and Portfolio Construction
data['Signal'] = 0
# Simple moving average crossover strategy
data.loc[data['Adj Close'] > data['RollingMean'], 'Signal'] = 1
data.loc[data['Adj Close'] < data['RollingMean'], 'Signal'] = -1

# Portfolio returns
data['StrategyReturns'] = data['Signal'].shift(1) * data['Returns']

In [ ]:
# Phase 4: Vectorized Backtest
# Calculate cumulative returns
data['CumulativeMarketReturns'] = (1 + data['Returns']).cumprod()
data['CumulativeStrategyReturns'] = (1 + data['StrategyReturns']).cumprod()

In [ ]:
# Phase 5: Performance Metrics
sharpe_ratio = data['StrategyReturns'].mean() / data['StrategyReturns'].std() * np.sqrt(252)
sortino_ratio = data['StrategyReturns'].mean() / data[data['StrategyReturns'] < 0]['StrategyReturns'].std() * np.sqrt(252)
max_drawdown = (data['CumulativeStrategyReturns'].cummax() - data['CumulativeStrategyReturns']).max()
calmar_ratio = data['StrategyReturns'].mean() * 252 / max_drawdown

print(f"Sharpe Ratio: {sharpe_ratio}")
print(f"Sortino Ratio: {sortino_ratio}")
print(f"Calmar Ratio: {calmar_ratio}")
print(f"Max Drawdown: {max_drawdown}")

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['CumulativeMarketReturns'], label='Market Returns')
plt.plot(data['CumulativeStrategyReturns'], label='Strategy Returns')
plt.title('Equity Curve')
plt.legend()
plt.show()

In [ ]:
# Phase 6: Monitoring Stub
# Placeholder for future monitoring code
print("Monitoring phase is under development.")